## Calendar Class
An instance of personal calendar, with tasks assigned to blocks.

In [41]:
from gurobipy import Model, GRB, quicksum
import numpy as np
    
class Calendar: 
    def __init__(self, task_domains, task_names, priorities=None, time_horizon=None):
        self.task_domains = task_domains # 2D array
        self.task_names = task_names
        if time_horizon is not None:
            self.time_horizon = time_horizon
        else:
            self.time_horizon = max(t for domain in task_domains for t in domain) # set max
        if priorities is not None:
            self.priorities = priorities
        else:
            self.priorities = [1 for _ in range(len(task_times))]

    """
    Returns an array of times that contain tasks already
    """
    def get_calendar(self):
        task_domains = self.task_domains  
        H = self.time_horizon         
        all_blocked = {i for domain in task_domains for i in domain}
        times_blocked = [1 if i in all_blocked else 0 for i in range(H)]
        return times_blocked


    """
    Adds tasks to this calendar
    """
    def add_tasks(self, tasks_dict):
        for task_name, task_start_end in tasks_dict.items():
            self.task_domains.append([i for i in range(task_start_end[0], task_start_end[1] + 1)])


## Optimizer Class

In [42]:
class PSC_Optimizer:
    def __init__(self, task_times, deadlines, task_names, current_calendar, priorities=None, priority_constant=0.15):
        self.task_times = task_times  # Array (length num tasks)
        self.deadlines = deadlines  # Array (length num tasks)
        self.task_names = task_names
        self.num_tasks = len(task_times)
        self.current_calendar = current_calendar # NOTE: TIME BLOCKS MUST MATCH UP TO THIS OPTIMIZER'S TIME BLOCKS
        if priorities is not None:
            self.priorities = priorities
        else:
            self.priorities = [1 for _ in range(len(task_times))]
        self.priority_constant = priority_constant # MUST BE NEGATIVE
        self.max_time = max(deadlines, current_calendar.time_horizon)  # Maximum available time


    def OptimizeCalendar(self):
        if len(self.task_times) != len(self.deadlines):
            raise ValueError("Mismatch in task_times and deadlines length")

        # Create a new model
        model = Model("PSC-MIP-V2")

        # Decision Variables
        task_to_block = model.addVars(self.max_time, self.num_tasks, vtype=GRB.BINARY, name="x")  # Task assigned to time block
        task_starting_time = model.addVars(self.max_time, self.num_tasks, vtype=GRB.BINARY, name="t")  # Task start indicator

        
        # Define dicts for task times and deadlines
        task_times = {j: self.task_times[j] for j in range(self.num_tasks)}  # All tasks take task_times[j] time blocks
        deadlines = {j: self.deadlines[j] for j in range(self.num_tasks)}  # Deadline is deadlines[j] for each block

        # Decision Variables
        task_to_block = model.addVars(self.max_time, self.num_tasks, vtype=GRB.BINARY, name="x")  # x[i, j] binary
        task_starting_time = model.addVars(self.max_time, self.num_tasks, vtype=GRB.BINARY, name="t")  # t[i, j] binary
        anti_anxiety = model.addVars(self.num_tasks, vtype=GRB.INTEGER, name="a")


        #Constraint 0: Cannot schedule tasks in blocks that already have something scheduled
        unavailable_blocks = self.current_calendar.get_calendar()
        for j in self.num_tasks:
            for i in range(len(unavailable_blocks)):
                if unavailable_blocks[i] > 0:
                    model.addConstr(task_to_block[i, j] == 0)

        # Constraint 1: Ensure each task is assigned exactly `task_times[j]` blocks in the time horizon
        for j in range(self.num_tasks):
            model.addConstr(
                sum(task_to_block[i, j] for i in range(self.max_time)) == self.task_times[j],
                f"Sum_x_{j}"
            )

        # Constraint 2: Each task must take place in the blocks between its starting time and its dealine
        for i in range(self.max_time):
            model.addConstr(i*task_starting_time[i, j] <= (self.deadlines[j] - self.task_times[j]), f"Bound_t_{i}_{j}")

        # Constraint 2.5: Tasks cannot start on the same block
        for i in range(self.max_time):
            model.addConstr(sum(task_starting_time[i, j] for j in range(self.num_tasks)) <= 1, f"One starting time per time block")

        # Constraint 3: At most 1 task per time block
        for i in range(self.max_time):
            model.addConstr(sum(task_to_block[i, j] for j in range(self.num_tasks)) <= 1, f"One task per time block")

        # Constraint 4: Each task has exactly one starting time
        for j in range(self.num_tasks):
            model.addConstr(
                sum(task_starting_time[i, j] for i in range(self.deadlines[j])) == 1,
                f"Unique_t_{j}"
            )

        # Constraint 5: Linking task_starting_time with task_to_block
        for j in range(self.num_tasks):
            latest_start = min(self.max_time - task_times[j], deadlines[j] - task_times[j])
            for i in range(latest_start + 1):  # Include latest valid start
                for k in range(task_times[j]):
                    model.addConstr(
                        task_to_block[i + k, j] >= task_starting_time[i, j],
                        f"Start_link_{i}_{j}_{k}"
                    )

        # Constraint 6: (Consecutivity) If a task starts at time t, then it occupies time blocks t to t + d[j] - 1
        model.addConstrs(
            task_to_block[t_prime, j] >= task_starting_time[t, j]
            for j in range(self.num_tasks)
            for t in range(self.max_time - self.deadlines[j] + 1)
            for t_prime in range(t, t + self.task_times[j])
        )
    
        
        # Temporary Objective: Minimize task starting times
        model.setObjective(
            sum(sum((1+i)*task_starting_time[i, j] * (1 - self.priority_constant * self.priorities[j]) 
            for j in range(self.num_tasks)) 
            for i in range(self.max_time)),
            GRB.MINIMIZE
        )
        
        
        # Solve the model
        model.optimize()


        task_to_times_array = np.zeros((self.max_time, self.num_tasks), dtype=int)

        for i in range(self.max_time):
            for j in range(self.num_tasks):
                task_to_times_array[i, j] = task_to_block[i, j].X

        start_times_array = np.zeros((self.max_time, self.num_tasks), dtype=int)

        for i in range(self.max_time):
            for j in range(self.num_tasks):
                start_times_array[i, j] = task_starting_time[i, j].X

        print(f"=======task_to_times: ======= \n {task_to_times_array}")
        print(f"=======start_times: ======= \n {start_times_array}")


        # Store results
        results_dict = {}


        if model.status == GRB.OPTIMAL:
            print("Optimal Solution Found:")
            for j in range(self.num_tasks):
                for i in range(self.max_time):
                    if task_starting_time[i, j].x == 1:  # Check if task starts here
                        start_time = i
                        end_time = i + self.task_times[j]
                        results_dict[self.task_names[j]] = (start_time, end_time)
                        print(f"Task {self.task_names[j]} scheduled from {start_time} to {end_time}")

        else:
            print("No optimal solution found.")

        return results_dict

## Testing

In [43]:
test_suites = dict()

# Instance Test 1
task_times_1 = [2, 2, 2]
deadlines_1 = [30, 30, 30]
task_names_1 = ["task1", "task2", "task3"]
priorities_1 = [4, 2, 3]
test_suites[1] = [task_times_1, deadlines_1, task_names_1, priorities_1]

# Instance Test 2
task_times_2 = [3, 5, 7, 2]
deadlines_2 = [4, 10, 20, 10]
task_names_2 = ["task1", "task2", "task3", "task4"]
test_suites[2] = [task_times_2, deadlines_2, task_names_2]
priorities_2 = [1, 1, 1, 5, 1]


# Create instance
cal_1 = PSC_Optimizer(task_times_1, deadlines_1, task_names_1, priorities_1)
cal_2 = PSC_Optimizer(task_times_2, deadlines_2, task_names_2, priorities_2)


# Optimize
print(cal_1.OptimizeCalendar())
# print(cal_2.OptimizeCalendar())


AttributeError: 'list' object has no attribute 'time_horizon'